In [28]:
from datasets import load_dataset
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import BartForConditionalGeneration, BartTokenizer, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
import requests 
from bs4 import BeautifulSoup 
import pandas as pd
from tqdm import tqdm

### Context Based Dataset

In [29]:
dataset_1 = load_dataset("9wimu9/eli5_mult_answers_en_no_answer_in_context") # in deep
dataset_2 = load_dataset("mlxen/squad_1_1_smallcase_context") #one word
dataset_3 = load_dataset("nbtpj/multi-context-long-answer-dataset") # in short
dataset_8 = load_dataset("LasRuinasCirculares/sft_correct")
dataset_9 = load_dataset("robbiegwaldd/rephrase_train") # all

In [30]:
dataset_1

DatasetDict({
    train: Dataset({
        features: ['question', 'contexts', 'gold_answer'],
        num_rows: 71236
    })
    test: Dataset({
        features: ['question', 'contexts', 'gold_answer'],
        num_rows: 7916
    })
})

In [31]:
agent_1 = dataset_1["train"]["question"]
raw_context = dataset_1["train"]["contexts"]
modified_context = []
for c in raw_context:
    final_string = ""
    for i in c:
        final_string = final_string+" "+ i
    modified_context.append(final_string)
context = modified_context
agent_2 = dataset_1["train"]["gold_answer"]

In [32]:
agent_1.extend(dataset_2["train"]["question"])
context.extend(dataset_2["train"]["context"])
agent_2.extend([i["text"][0] for i in dataset_2["train"]["answers"]])

agent_1.extend([i.split("Question:")[1].replace("Answer:","") for i in dataset_8["train"]["instruction"]])
context.extend([i.replace("Evidence:","").replace("**","").split("Question:")[0].replace("\n","") for i in dataset_8["train"]["instruction"]])
agent_2.extend(dataset_8["train"]["output"])

In [33]:
channelNames =["pubmed_qa_X_squad_num_channel_1_test","pubmed_qa_X_squad_num_channel_1_train","pubmed_qa_X_squad_num_channel_2_test","pubmed_qa_X_squad_num_channel_2_train","pubmed_qa_X_squad_num_channel_3_test","pubmed_qa_X_squad_num_channel_3_train","pubmed_qa_X_squad_num_channel_4_test","pubmed_qa_X_squad_num_channel_4_train","pubmed_qa_num_channel_1_test","pubmed_qa_num_channel_1_train","pubmed_qa_num_channel_2_test","pubmed_qa_num_channel_2_train","pubmed_qa_num_channel_3_test","pubmed_qa_num_channel_3_train","pubmed_qa_num_channel_4_test","pubmed_qa_num_channel_4_train","squad_num_channel_1_test","squad_num_channel_1_train","squad_num_channel_2_test","squad_num_channel_2_train","squad_num_channel_3_test","squad_num_channel_3_train","squad_num_channel_4_test","squad_num_channel_4_train"]

In [34]:
question = []
cont = []
answer = []
for channel in channelNames:
    question.extend([i[0].split("<||||>")[0] for i in dataset_3[channel]["context"]])
    cont.extend([i[0].split("<||||>")[1] for i in dataset_3[channel]["context"]])
    answer.extend([i for i in dataset_3[channel]["answer"]])
temp_df = pd.DataFrame()
temp_df["question"] = question
temp_df["cont"] = cont
temp_df["answer"] = answer
df = temp_df.drop_duplicates(subset=['question','cont','answer'], keep='first')
agent_1.extend(df["question"].tolist())
context.extend(df["cont"].tolist())
agent_2.extend(df["answer"].tolist())

In [35]:
datasets = ["bart-base-squad.end2end.amazon","bart-base-squad.end2end.new_wiki","bart-base-squad.end2end.nyt","bart-base-squad.end2end.reddit","bart-base-squad.multitask.amazon","bart-base-squad.multitask.new_wiki","bart-base-squad.multitask.nyt","bart-base-squad.multitask.reddit","bart-base-squad.pipeline.amazon","bart-base-squad.pipeline.new_wiki","bart-base-squad.pipeline.nyt","bart-base-squad.pipeline.reddit","bart-base-squad.qg_reference.amazon","bart-base-squad.qg_reference.new_wiki","bart-base-squad.qg_reference.nyt","bart-base-squad.qg_reference.reddit","bart-large-squad.end2end.amazon","bart-large-squad.end2end.new_wiki","bart-large-squad.end2end.nyt","bart-large-squad.end2end.reddit","bart-large-squad.multitask.amazon","bart-large-squad.multitask.new_wiki","bart-large-squad.multitask.nyt","bart-large-squad.multitask.reddit","bart-large-squad.pipeline.amazon","bart-large-squad.pipeline.new_wiki","bart-large-squad.pipeline.nyt","bart-large-squad.pipeline.reddit","bart-large-squad.qg_reference.amazon","bart-large-squad.qg_reference.new_wiki","bart-large-squad.qg_reference.nyt","bart-large-squad.qg_reference.reddit","t5-base-squad.end2end.amazon","t5-base-squad.end2end.new_wiki","t5-base-squad.end2end.nyt","t5-base-squad.end2end.reddit","t5-base-squad.multitask.amazon","t5-base-squad.multitask.new_wiki","t5-base-squad.multitask.nyt","t5-base-squad.multitask.reddit","t5-base-squad.pipeline.amazon","t5-base-squad.pipeline.new_wiki","t5-base-squad.pipeline.nyt","t5-base-squad.pipeline.reddit","t5-base-squad.qg_reference.amazon","t5-base-squad.qg_reference.new_wiki","t5-base-squad.qg_reference.nyt","t5-base-squad.qg_reference.reddit","t5-large-squad.end2end.amazon","t5-large-squad.end2end.new_wiki"]

In [36]:
for i in tqdm(range(0,len(datasets))):
    # print(f'\rDataset Name: {dataset}', end='', flush=True)
    ds = load_dataset("lmqg/qa_squadshifts_synthetic", datasets[i])
    agent_1.extend(i for i in ds["train"]["question"])
    context.extend(i for i in ds["train"]["context"])
    agent_2.extend(i["text"][0] for i in ds["train"]["answers"])

100%|██████████| 50/50 [01:00<00:00,  1.21s/it]


In [37]:
dataset_9

DatasetDict({
    train: Dataset({
        features: ['id', 'original_snippet', 'rephrased_snippet'],
        num_rows: 291032
    })
})

In [38]:
len(agent_1)

1469571

In [39]:
questions = []
answers = []
conte = []
split_qa = [i.split("\n\n") for i in dataset_9["train"]["rephrased_snippet"]]
cont = [i.replace("\n","") for i in dataset_9["train"]["original_snippet"]]
for i in tqdm(range(0,len(split_qa))):
    q = [];a = [];c = []
    if "\nAnswer" not in split_qa[i][0]:
        if len(split_qa[i]) % 2 != 0:
            split_qa[i] = split_qa[i][:-1]
        for j in range(0,len(split_qa[i]),2):
            q.append(split_qa[i][j])
            a.append(split_qa[i][j+1])
            c.append(cont[i])
    else:
        new_split = [j.split("\n") for j in split_qa[i]]
        for j in range(0,len(new_split)):
            if len(new_split[j]) %2 !=0:
                continue
            q.append(new_split[j][0].replace("Question:",""))
            a.append(new_split[j][1].replace("Answer:",""))
            c.append(cont[i])
    answers.extend(a)
    questions.extend(q)
    conte.extend(c)
    # print(len(answers),len(questions))
    # print(split_qa[i])
agent_1.extend(questions)
context.extend(conte)
agent_2.extend(answers)

100%|██████████| 291032/291032 [00:00<00:00, 369566.84it/s]


In [40]:
len(agent_1)

1802143

In [41]:
len(context)

1802143

In [42]:
len(agent_2)

1802143

In [43]:
temp_df = pd.DataFrame()
temp_df["agent_1"] = agent_1
temp_df["context"] = context
temp_df["agent_2"] = agent_2

In [44]:
temp_df.to_csv("../Data/CleanedDatasets/ContextBasedQuestions.csv")

In [45]:
temp_df

,agent_1,context,agent_2
0,why do the healthiest of foods cause bad gas?,Actually it is because most of people eat tha...,These foods contain certain sugars that our bo...
1,"In layman's terms, what would actually happen ...",The decay of false vacuum is a fascinating to...,I'd answer the question but the default reddit...
2,"When I move at speed through the air, it's col...",The spacecraft re-enters through the atmosphe...,There are (at least) 3 different competing mec...
3,what is the big deal about russia invading ukr...,National Sovereignty is a very big deal so an...,Essentially the events in Crimea threaten to d...
4,why are there different types of helmets if yo...,Most helmets are pretty similar in basic cons...,To have as little protection as is safe for th...
...,...,...,...
1802138,Question: Can you convert this paragraph into ...,Creating ThemesThe game room is where you are ...,Here are two images illustrating how the game ...
1802139,Game room colors 1,Creating ThemesThe game room is where you are ...,Notice that every even gameboard units are hid...
1802140,Game room colors 2,Creating ThemesThe game room is where you are ...,The background image in this example could be ...
1802141,Just load the MMO board game client software t...,Creating ThemesThe game room is where you are ...,The client software has 8 chess sets to choose...


### Google Search

In [193]:
from googlesearch import search
def get_google_search_links(query):
    return [link for link in search(query)]

#### Normal Search Queries

In [194]:
website_links = get_google_search_links('To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?')

In [195]:
website_links[0]

'https://en.wikipedia.org/wiki/Lourdes_apparitions'

In [196]:
URL = website_links[0] 
r = requests.get(URL) 
soup = BeautifulSoup(r.content, 'html5lib')

In [197]:
raw_content = soup.find_all(["p","h"])
filtered_content = [i.text for i in raw_content]
filtered_content

['The Lourdes apparitions are several Marian apparitions reported in 1858 by Bernadette Soubirous, the 14-year-old daughter of a miller, in the town of Lourdes in Southern France.\n',
 'From 11 February to 16 July 1858, she reported 18 apparitions of "a Lady". Soubirous described the lady as wearing a white veil and a blue girdle; she had a golden rose on each foot and held a rosary of pearls. After initial skepticism from the local clergy, these claims were eventually declared to be worthy of belief by the Catholic Church after a canonical investigation. The apparition is known as Our Lady of Lourdes.\n',
 'According to Soubirous, her visions occurred at the grotto of Massabielle, just outside Lourdes. On 16 July 1858, Soubirous visited the grotto for the last time and said: "I have never seen her so beautiful before."[2][page\xa0needed] On 18 January 1862, the local bishop declared: "The Virgin Mary did appear indeed to Bernadette Soubirous."[3] Soubirous was canonized a saint in 193

#### Reasearch Paper Queries

In [198]:
website_links = get_google_search_links('what is backtesting in algorithmic trading')

In [199]:
website_links

['https://medium.com/@anrodz/algo-trading-backtesting-your-algorithm-bd6d7385c89c',
 'https://www.investopedia.com/terms/b/backtesting.asp',
 'https://utradealgos.com/blog/what-is-algo-backtesting-and-how-does-it-work/',
 'https://speedbot.tech/blog/speedbot-1/future-of-backtesting-in-algo-trading-what-to-know-41',
 'https://dataman-ai.medium.com/practical-algorithmic-trading-2-backtesting-19971c8a89bd',
 'https://zorro-project.com/backtest.php',
 'https://blog.quantinsti.com/backtesting/',
 'https://www.linkedin.com/pulse/what-backtesting-why-important-algorithmic-trading-algoedge-cad-lcp7e',
 'https://www.udemy.com/course/mastering-backtesting-for-algorithmic-trading/',
 'https://www.linkedin.com/pulse/what-backtesting-why-important-algorithmic-trading-algoedge-cad-lcp7e',
 'https://www.udemy.com/course/mastering-backtesting-for-algorithmic-trading/',
 'https://tradetron.tech/blog/backtest/']

In [200]:
URL = website_links[2] 
r = requests.get(URL) 
soup = BeautifulSoup(r.content, 'html5lib')
raw_content = soup.find_all(["p","h"])
filtered_content = [i.text for i in raw_content]
filtered_content

['uTrade Algos',
 'Home » Blogs » What Is Algo Backtesting and How Does It Work?',
 'In the fast-paced world of financial markets, traders and investors are constantly seeking an edge to stay ahead of the curve. Algorithmic trading, or algo trading, has emerged as a powerful tool to gain this competitive advantage. It involves the use of automated trading systems, often fuelled by complex algorithms, to execute trades with speed and precision. However, before deploying these algorithms in live markets, algo traders employ a crucial step to evaluate and fine-tune their strategies: backtesting. Here, we will find out what algo backtesting is and how it works.\xa0',
 'Table of Contents',
 'Algo backtesting is a systematic process that allows traders to assess the performance of their trading strategies using historical data. By simulating past market conditions and applying their trading rules to historical price and volume data, traders can gain valuable insights into how their strategie